# DentVLM PAN training-aligned baseline

Full PAN image ? 12 canonical independent questions ? native responses ? deterministic parsing ? supported findings and available locations ? validated dentist report.

The baseline is `pan_training_aligned_v1`. All 12 tasks run regardless of benchmark labels. Patient laterality remains unresolved. Default counts are disabled. Start with five images per dataset, inspect saved prompts, runtime checks and responses, then set `LIMIT = None` for the available splits. Prompt alignment does not establish clinical performance or native-runtime equivalence.

In [ ]:
# ============================================================
# CELL 1 - Python dependencies (llama.cpp is compiled later with CUDA)
# ============================================================
%pip install -q "huggingface_hub>=0.26" "openai>=1.55" "requests>=2.31" "pillow>=10.0" "pandas>=1.5"
print("Python dependencies installed.")

In [ ]:
# ============================================================
# CELL 2 - Locate and import the project files
# ============================================================
import os, sys, json, shutil, subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/dental_x-ray")  # folder holding the project .py files
REQUIRED_PROJECT_FILES = {"dental_pipeline.py", "dental_eval.py", "llama_runtime.py", "location_adapter.py",
                          "llm_api.py", "llm_parser.py", "report_writer.py", "dental_analysis.py",
                          "experiments.py", "run_monitor.py", "response_cache.py"}
if not PROJECT_DIR.is_dir() and Path.cwd().joinpath("dental_pipeline.py").is_file():
    PROJECT_DIR = Path.cwd()
missing = [f for f in REQUIRED_PROJECT_FILES if not (PROJECT_DIR / f).is_file()]
if missing:
    raise FileNotFoundError(f"Missing project files in {PROJECT_DIR}: {missing}")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import dental_pipeline as dp
import dental_eval as ev
import dental_analysis as da
import location_adapter as la
import llm_api
import llm_parser as lp
import run_monitor as mon
import experiments as xp
import report_writer as rw
from llama_runtime import LlamaCppServer, build_llama_cpp, convert_to_gguf, download_gguf, local_gguf
print("PROJECT_DIR =", PROJECT_DIR)
print("Project imports succeeded.")

In [ ]:
# CELL 3 - Fixed PAN baseline configuration
CALL_LOG = "auto"
OUTPUT_ROOT = "/kaggle/working/dentvlm_pan_training_aligned_v1"
PROVIDERS = {
    "openai": {"base_url": "https://api.openai.com/v1",
               "api_key": llm_api.secret("OPENAI_API_KEY", required=False)},
    "openrouter": {"base_url": "https://openrouter.ai/api/v1",
                   "api_key": llm_api.secret("OPENROUTER_API_KEY", required=False)},
    "gemini": {"base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
               "api_key": llm_api.secret("GEMINI_API_KEY", required=False)},
}
llm_api.configure_providers(PROVIDERS)
mon.CALL_LOG = CALL_LOG


LIMIT = 5  # smoke validation first; None evaluates the available full splits
LOCATION_TRUTH = "geometry"  # UMFIH: approximate secondary truth; DENTEX: original FDI first
SHARED = {"output_root": OUTPUT_ROOT, "location_truth": LOCATION_TRUTH,
          "evaluate_location": True, "counting": False, "parser_mode": "code",
          "report_images": 5, "location_failure_policy": "exclude"}
EXPERIMENTS = xp.build([{"name": dp.PROFILE}], shared=SHARED)
xp.show(EXPERIMENTS)
DATASETS = [
    {"name": "umfih_test", "kind": "yolo", "limit": LIMIT,
     "images": "/kaggle/working/umfih_14class/data/test/images",
     "labels": "/kaggle/working/umfih_14class/data/test/labels"},
    {"name": "dentex_val", "kind": "dentex", "limit": LIMIT,
     "images": "/kaggle/working/DENTEX/validation_data/quadrant_enumeration_disease/xrays",
     "annotations": "/kaggle/working/DENTEX/validation_triple.json"},
]

def new_questions_per_image(configs):
    return [(cfg["name"], len(xp.protocol(cfg).tasks()), len(xp.protocol(cfg).tasks())) for cfg in configs]

for name, questions, _ in new_questions_per_image(EXPERIMENTS):
    print(f"{name}: {questions} canonical questions per image; sequential single-slot execution")
print("Transport retries repeat identical requests; unreadable answers remain unresolved.")
LLAMA_CPP_DIR = "/kaggle/working/llama.cpp"
LLAMA_CPP_REF = "b10516"
SERVER_HOST, SERVER_PORT, SERVER_ALIAS = "127.0.0.1", 8080, "dentvlm"
SERVER_LOG_PATH = "/kaggle/working/llama_dentvlm_server.log"
SERVER_STARTUP_TIMEOUT = 300.0
MODEL_DIR = "/kaggle/working/models/dentvlm_pan_v1"
CONVERT_WORK_DIR = "/tmp/dentvlm_hf"  # scratch for the 17 GB safetensors; not persisted
HF_TOKEN_SECRET = "HF_TOKEN"          # Kaggle secret holding a Hugging Face token (or set env HF_TOKEN)
CUDA_ARCH = None                      # None = auto-detect (P100 fallback 60)
BUILD_JOBS = 4

LOCAL = [c for c in EXPERIMENTS if xp.is_local(c)]
print(f"\ndatasets = {[d['name'] for d in DATASETS]} | local experiments = {[c['name'] for c in LOCAL]}")

In [ ]:
# ============================================================
# CELL 4 - Environment diagnostics
# ============================================================
import platform
print("Python:", platform.python_version(), "|", platform.platform())


def detect_cuda_arch(fallback="60"):
    try:
        output = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                                         text=True, stderr=subprocess.STDOUT)
        arch = output.strip().splitlines()[0].strip().replace(".", "")
        if arch.isdigit():
            return arch
    except Exception as exc:
        print("CUDA architecture auto-detection failed:", exc)
    print(f"Falling back to CUDA architecture {fallback}.")
    return fallback


CUDA_ARCH_RESOLVED = None
if LOCAL:
    for executable in ("git", "cmake", "nvcc", "nvidia-smi"):
        print(f"{executable:12s}:", shutil.which(executable))
    if shutil.which("nvidia-smi"):
        subprocess.run(["nvidia-smi"], check=False)
    for executable in ("cmake", "git", "nvcc"):
        if not shutil.which(executable):
            raise RuntimeError(f"{executable} is required to build llama.cpp; enable a GPU accelerator.")
    CUDA_ARCH_RESOLVED = str(CUDA_ARCH) if CUDA_ARCH else detect_cuda_arch("60")
    print("CUDA_ARCH_RESOLVED =", CUDA_ARCH_RESOLVED)
else:
    print("No local experiment; llama.cpp checks skipped.")

In [ ]:
# ============================================================
# CELL 5 - Build or find the pinned llama.cpp server
# ============================================================
LLAMA_SERVER = None
if LOCAL:
    # Reuses an existing build only if it was built from LLAMA_CPP_REF; otherwise rebuilds.
    LLAMA_SERVER = Path(build_llama_cpp(source_dir=LLAMA_CPP_DIR, cuda_arch=CUDA_ARCH_RESOLVED,
                                        jobs=BUILD_JOBS, ref=LLAMA_CPP_REF)).resolve()
    print("llama-server =", LLAMA_SERVER)
else:
    print("No local experiment; llama.cpp build skipped.")

In [ ]:
# ============================================================
# CELL 6 - Local DentVLM: the GGUF files, and one llama.cpp server at a time
# ============================================================
# DentVLM (Hugging Face ZJU-AI4H/DentVLM, gated with automatic approval, CC BY-NC 4.0) ships as bf16
# safetensors; model_source "convert" downloads and converts it once (~17 GB scratch, ~9.5 GB kept),
# "local" uses files already in MODEL_DIR (an attached Kaggle dataset), "hf" downloads them from your own
# gguf_repo_id. Experiments asking for the same files convert or download once; experiments asking for the
# same server settings share the running server.
import requests

MODELS = {}
for cfg in LOCAL:
    key = xp.model_key(cfg)
    if key in MODELS:
        continue
    source, repo_id, model_filename, mmproj_filename, hf_revision, gguf_revision = key
    hf_token = llm_api.secret(HF_TOKEN_SECRET, required=False)
    # A conversion or download that fails takes only the experiments needing these files; the error is in
    # the ledger and the sweep reports them as failed instead of dying here.
    with mon.guard(f"model files {model_filename}", LEDGER) as step:
        if source == "convert":
            if not hf_token:
                print(f"No Hugging Face token under {HF_TOKEN_SECRET!r}; only model_source 'local' works without one.")
            MODELS[key] = convert_to_gguf(MODEL_DIR, LLAMA_CPP_DIR, hf_token=hf_token, work_dir=CONVERT_WORK_DIR,
                                          model_filename=model_filename, mmproj_filename=mmproj_filename, revision=hf_revision)
            print("Keep both GGUF files and dentvlm_provenance.json (private Kaggle dataset or Hugging Face repo) and switch "
                  "model_source to 'local' or 'hf' for the next session.")
        elif source == "local":
            MODELS[key] = local_gguf(MODEL_DIR, model_filename, mmproj_filename)
        else:
            MODELS[key] = download_gguf(MODEL_DIR, repo_id, model_filename, mmproj_filename, hf_token, revision=gguf_revision)
    if not step.ok:
        continue
    for label, path in (("Language model", MODELS[key].model_path), ("Vision projector", MODELS[key].mmproj_path)):
        print(f"{label}: {path} ({Path(path).stat().st_size / 1024**3:.2f} GiB)")

SERVER = SERVER_SETTINGS = None


def local_server(cfg):
    """The llama.cpp server for one experiment, restarted only when its local settings change."""
    global SERVER, SERVER_SETTINGS
    if SERVER is not None and SERVER_SETTINGS == xp.server_key(cfg):
        return SERVER
    if SERVER is not None:
        SERVER.stop()
    files = MODELS[xp.model_key(cfg)]
    SERVER = LlamaCppServer(binary=LLAMA_SERVER, model_path=files.model_path, mmproj_path=files.mmproj_path,
                            host=SERVER_HOST, port=SERVER_PORT, alias=SERVER_ALIAS,
                            n_gpu_layers=cfg["n_gpu_layers"], ctx_size=cfg["ctx_size"],
                            image_max_tokens=cfg["image_max_tokens"], image_min_tokens=cfg["image_min_tokens"],
                            startup_timeout=SERVER_STARTUP_TIMEOUT, log_path=SERVER_LOG_PATH)
    SERVER.start(reuse_existing=True)
    ids = [m.get("id") for m in requests.get(f"{SERVER.base_url}/v1/models", timeout=10).json().get("data", [])]
    if SERVER_ALIAS not in ids:
        raise RuntimeError(f"Expected alias {SERVER_ALIAS!r}; /v1/models returned {ids}. See {SERVER_LOG_PATH}.")
    SERVER_SETTINGS = xp.server_key(cfg)
    print(f"{cfg['name']}: DentVLM server verified at {SERVER.base_url}")
    return SERVER


def open_runner(cfg):
    """The analyzer of one experiment: its hosted model, or the local server started above."""
    return xp.runner(cfg, local_server(cfg) if xp.is_local(cfg) else None)


# One reader per experiment, built once and shared by its analyzer run, its location adapter and its
# report writer, so every parser call of that experiment is counted once and one fingerprint says how
# the whole experiment read its replies.
PARSERS = {}


def open_parser(cfg):
    if cfg["name"] not in PARSERS:
        PARSERS[cfg["name"]] = xp.parser(cfg)
    return PARSERS[cfg["name"]]


def read_reply(parser, question, reply):
    answer = None if reply.get("truncated") or reply.get("error") else dp.extract_answer(reply["text"])
    regions = dp.extract_regions(reply["text"]) if answer == "yes" else []
    return answer, regions, answer is None or (answer == "yes" and not regions)


print("Model files ready:", len(MODELS), "| local server and parser helpers defined")

In [ ]:
# ============================================================
# CELL 7 - Load ground truth for every dataset (no model calls)
# ============================================================
# Each dataset is loaded behind a guard: a missing folder or a malformed label file prints in full and the
# other datasets still load, and a dataset that fails is dropped from DATASETS so the cells below stay
# consistent. truth_report prints what was loaded and anything unusable in it (images with no label file,
# labels with no image, annotated images missing from disk) before a single model call is paid for.
GT = {}
for spec in DATASETS:
    with mon.guard(f"load {spec['name']}", LEDGER) as step:
        if spec["kind"] == "yolo":
            gt = ev.load_yolo(spec["images"], spec["labels"])
        elif spec["kind"] == "dentex":
            gt = ev.load_dentex(spec["images"], spec["annotations"])
        else:
            raise ValueError(f"unknown dataset kind {spec['kind']!r}")
        if spec.get("limit"):
            gt = dict(sorted(gt.items())[: spec["limit"]])
        GT[spec["name"]] = gt
        ev.truth_report(gt, spec["name"])
DATASETS = [d for d in DATASETS if d["name"] in GT]
if not DATASETS:
    raise RuntimeError("no dataset loaded; fix the paths reported above and rerun this cell")
print("datasets ready:", [d["name"] for d in DATASETS])

In [ ]:
# CELL 8 - Five-image smoke validation per dataset, all 12 tasks
# Saves complete artifacts in the baseline directory; CELL 9 resumes them without new calls.
SMOKE = {}
for cfg in EXPERIMENTS:
    with mon.guard(f"smoke {cfg['name']}", LEDGER) as step:
        runner, parser = open_runner(cfg), open_parser(cfg)
        print("Verified runtime:", json.dumps(runner.settings(), indent=1))
        for dataset, truth in GT.items():
            selected = dict(sorted(truth.items())[:cfg["smoke_images"]])
            if not selected:
                continue
            dp.run_dataset(runner, {i: e["path"] for i,e in selected.items()}, xp.run_dir(cfg, dataset),
                           protocol=xp.protocol(cfg), parser=parser, resume=True,
                           provenance=xp.provenance(cfg, llama_cpp_ref=LLAMA_CPP_REF))
            saved = dp.load_results(xp.run_dir(cfg, dataset))
            SMOKE[dataset] = {i: saved[i] for i in selected if i in saved}
            for image_id, result in SMOKE[dataset].items():
                print(dataset, image_id, dp.result_line(result))
                for call in result["calls"]:
                    print(call["task"], call["question"], "finish:", call.get("finish_reason"))
                    print(call["text"])
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_ROOT, "smoke.json").write_text(json.dumps(SMOKE, indent=1), encoding="utf8")

In [ ]:
# CELL 8.5 - Optional full PAN analysis for another image
# The same fixed contract is used; arbitrary prompts and wording variants are retired.
CUSTOM_IMAGE_PATH = None
CUSTOM_INFERENCE_RESULT = None
if CUSTOM_IMAGE_PATH:
    cfg = EXPERIMENTS[0]
    CUSTOM_INFERENCE_RESULT = dp.analyze_image(open_runner(cfg), CUSTOM_IMAGE_PATH, xp.protocol(cfg))
    print(dp.dentist_report(CUSTOM_INFERENCE_RESULT))

In [ ]:
# ============================================================
# CELL 9 - Run every experiment (resumable: finished images are skipped)
# ============================================================
# One experiment at a time, one directory each: <output_root>/<name>/<dataset>/. The resolved configuration
# is saved as experiment.json and hashed into the run manifest, so a changed knob can never be mixed into a
# resumed run. A failure prints in full, is recorded in LEDGER and stepped over: one bad image never costs
# the rest of the dataset (a rerun resumes it) and one bad experiment never costs the sweep. Three failed
# images in a row stop that dataset instead, because that is a dead server rather than a bad image.

FAILED = {}
for cfg in EXPERIMENTS:
    name = cfg["name"]
    print(f"\n{'=' * 78}\n=== {name}: {xp.analyzer_name(cfg)} | {xp.protocol(cfg)}\n{'=' * 78}")
    with mon.guard(f"run {name}", LEDGER) as step:
        xp.record(cfg)
        runner, parser = open_runner(cfg), open_parser(cfg)
        print(f"  runner = {runner.settings()}")
        print("\n".join("  " + line for line in parser.summary_lines()))
        for spec in DATASETS:
            images = {image_id: g["path"] for image_id, g in GT[spec["name"]].items()}
            out = dp.run_dataset(runner, images, xp.run_dir(cfg, spec["name"]), protocol=xp.protocol(cfg),
                                 resume=True, ledger=LEDGER, parser=parser,
                                 provenance=xp.provenance(cfg, llama_cpp_ref=LLAMA_CPP_REF))
            print("  saved:", out)
    if not step.ok:
        FAILED[name] = step.error

print("\nfinished:", [c["name"] for c in EXPERIMENTS if c["name"] not in FAILED], "| failed:", sorted(FAILED))
LEDGER.report(path=Path(OUTPUT_ROOT) / "failures.json")

In [ ]:
# ============================================================
# CELL 10 - Location truth: translate ground-truth boxes into DentVLM's six cells (resumable)
# ============================================================
# Independent of the model run, and keyed by the adapter rather than by the experiment: every experiment
# using the same adapter reads the same translated boxes (whether they serve the location tables, the
# occupied-region counts, or both: the count target is one region per box, so the same truth) from
# <output_root>/location_truth/<dataset>/<adapter>/ instead of paying for them again.
# One JSON per image under boxes/, drawn images under drawn/ for audit; unparseable replies follow
# location_failure_policy, and every attempt and fallback is saved.
ADAPTED, TRUTH_DIRS = {}, {}
for cfg in EXPERIMENTS:
    name = cfg["name"]
    for spec in DATASETS:
        dataset = spec["name"]
        if all(b.get("fdi") for entry in GT[dataset].values() for b in entry["boxes"]):
            print(f"{dataset}: original FDI annotations; no location adapter calls")
            continue
        if cfg["location_truth"] == "geometry":
            print(f"{name}/{dataset}: fixed windows (and FDI tooth numbers where the dataset has them)")
            continue
        if not xp.uses_location_truth(cfg):
            print(f"{name}/{dataset}: neither location nor counts are scored; no true box is placed")
            continue
        serves = ("location and counts" if cfg["evaluate_location"] and cfg["counting"]
                  else "counts only (location scoring is off)" if cfg["counting"] else "location only")
        out = xp.truth_dir(cfg, dataset)
        if out in TRUTH_DIRS:
            ADAPTED[name, dataset] = TRUTH_DIRS[out]
            print(f"{name}/{dataset}: reuses {out} ({serves})")
            continue
        print(f"{name}/{dataset}: adapted truth serves {serves}")
        # A failed adapter leaves this pair out of ADAPTED, and the evaluation then scores it against the
        # fixed windows; the failure stays in the ledger so the fallback is never silent.
        with mon.guard(f"location {name}/{dataset}", LEDGER, note="scoring falls back to the fixed windows") as step:
            adapter = xp.location_adapter(cfg, parser=open_parser(cfg))
            TRUTH_DIRS[out] = ADAPTED[name, dataset] = la.adapt_dataset(adapter, GT[dataset], out, resume=True,
                                                                        ledger=LEDGER)
            print(f"{name}/{dataset}: {la.summarize(ADAPTED[name, dataset])}")
            agreement = ev.truth_agreement(GT[dataset], ADAPTED[name, dataset])
        if step.ok and agreement["boxes_with_fdi"]:
            # DENTEX carries FDI tooth numbers: exact cells, so this is the adapter's own accuracy.
            print(f"{name}/{dataset}: adapter vs FDI truth {agreement}")

In [ ]:
# ============================================================
# CELL 11 - Evaluate and compare every experiment
# ============================================================
import pandas as pd
from IPython.display import display

# Each experiment is scored against its own location truth and written to <name>/<dataset>/evaluation/, with
# its own evaluate_location and counting switches.
REPORTS = {}
for cfg in EXPERIMENTS:
    for spec in DATASETS:
        name, dataset = cfg["name"], spec["name"]
        # Scoring one experiment reads its saved results: a corrupt artifact or a dataset whose adapted
        # truth does not cover every image is reported and skipped, and the other experiments still rank.
        with mon.guard(f"score {name}/{dataset}", LEDGER) as step:
            results = dp.load_results(xp.run_dir(cfg, dataset))
            if not (set(GT[dataset]) & set(results)):
                LEDGER.note(f"score {name}/{dataset}", "no results yet; run CELL 9",
                            path=str(xp.run_dir(cfg, dataset)))
                continue
            truth = ev.apply_adapted(GT[dataset], ADAPTED[name, dataset]) if (name, dataset) in ADAPTED else GT[dataset]
            REPORTS[name, dataset] = ev.evaluate(truth, results, dataset=dataset,
                                                 out_dir=xp.run_dir(cfg, dataset) / "evaluation",
                                                 evaluate_location=cfg["evaluate_location"], counting=cfg["counting"])


# Primary exact/composite detection first; proxies and localization remain separate.
LEADERBOARD = pd.DataFrame([{**r["summary"], "experiment": name} for (name,dataset),r in REPORTS.items()])
FINDING_COMPARISON = pd.DataFrame([{**row, "experiment": name} for (name,dataset),r in REPORTS.items() for row in r["presence"]])
columns = ["dataset", "experiment", "FP", "false_positive_rate", "negative_scored", "precision", "predicted_positive",
           "TP", "sensitivity", "positive_scored", "unresolved_positive_truth", "unresolved_negative_truth",
           "resolved_coverage", "all_eligible_accuracy"]
if not LEADERBOARD.empty:
    display(LEADERBOARD[columns])
    LEADERBOARD.to_csv(Path(OUTPUT_ROOT) / "baseline_summary.csv", index=False)
if not FINDING_COMPARISON.empty:
    display(FINDING_COMPARISON[["dataset", "condition", "mapping_kind", "FP", "false_positive_rate", "negative_scored",
                                "precision", "predicted_positive", "TP", "sensitivity", "unparseable", "resolved_coverage"]])
print("Caries has benchmark truth. Calculus, Residual Crown and Insufficient Space for Primary Tooth Eruption do not; no accuracy is assigned.")
print("Periapical lesion vs Apical Periodontitis is a proxy, excluded from the primary summary.")
print("Patient laterality is unresolved. Source-frame localization is secondary and assumes the supplement mapping.")

# Optional historical comparison: existing artifacts are read without modification.
# Give complete compatible saved runs per dataset; image hashes are checked before pairing.
HISTORICAL_RUNS = {}  # e.g. {"umfih_test": {"historical": "/path/to/old/umfih_test"}}
COMPARISONS = {}
for dataset, historical in HISTORICAL_RUNS.items():
    cfg = EXPERIMENTS[0]
    runs = {**historical, cfg["name"]: xp.run_dir(cfg, dataset)}
    COMPARISONS[dataset] = da.compare_runs(GT[dataset], runs, dataset=dataset, evaluate_location=False, counting=False)
    ev.write_report(COMPARISONS[dataset], Path(OUTPUT_ROOT) / "comparison" / dataset)
    display(pd.DataFrame(COMPARISONS[dataset]["run_comparison"]))
LEDGER.report(path=Path(OUTPUT_ROOT) / "failures.json")

In [ ]:
# CELL 12 - Per-task and location diagnostics
INSPECT_EXPERIMENT = EXPERIMENTS[0]["name"]
INSPECT_DATASET = DATASETS[0]["name"]
report = REPORTS.get((INSPECT_EXPERIMENT, INSPECT_DATASET))
if report:
    for table in ("presence", "proxy_presence", "task_coverage", "regions", "parse_recovery", "call_usage"):
        if report.get(table):
            print(table)
            display(pd.DataFrame(report[table]))
    print("Unmentioned regions are not stated, never confirmed negative. Exact descriptor matches are retained for source-scoring reproduction.")

In [ ]:
# CELL 13 - Saved clinical summary and exact question/answer evidence
INSPECT_IMAGE = None
SHOW_RAW_CALLS = True
cfg = next(c for c in EXPERIMENTS if c["name"] == INSPECT_EXPERIMENT)
results = dp.load_results(xp.run_dir(cfg, INSPECT_DATASET))
if results:
    image_id = INSPECT_IMAGE or next(iter(sorted(results)))
    result = results[image_id]
    print(dp.dentist_report(result, counting=False))
    print("Patient laterality remains unresolved; do not choose a mapping by benchmark performance.")
    if SHOW_RAW_CALLS:
        for call in result["calls"]:
            print(call["task"], "| question:", call["question"], "| finish:", call.get("finish_reason"))
            print(call["text"])

In [ ]:
# CELL 14 - Validated report organization and deterministic clinical facts
# The writer sees only canonical IDs, statuses, and permitted locations.
# Full saved questions and answers remain in audit evidence. English clinical facts are rendered by code.
from IPython.display import Markdown, display

REPORT_EXPERIMENTS = [INSPECT_EXPERIMENT]  # one call per image per experiment; [c["name"] for c in EXPERIMENTS] for all

WRITTEN = {}
for cfg in [c for c in EXPERIMENTS if c["name"] in REPORT_EXPERIMENTS]:
    name = cfg["name"]
    with mon.guard(f"report writer {name}", LEDGER) as step:
        writer = xp.report_writer(cfg, parser=open_parser(cfg))
        print(f"{name}: report writer {writer.public()}")
    if not step.ok:
        continue
    for spec in DATASETS:
        dataset = spec["name"]
        with mon.guard(f"reports {name}/{dataset}", LEDGER) as step:
            results = dp.load_results(xp.run_dir(cfg, dataset))
            if not results:
                LEDGER.note(f"reports {name}/{dataset}", "no results to report; run CELL 9")
                continue
            WRITTEN[name, dataset] = rw.report_dataset(
                writer, results, xp.run_dir(cfg, dataset) / "reports" / writer.run_name,
                analyzer=xp.analyzer_name(cfg), resume=True, limit=cfg["report_images"], ledger=LEDGER)
            print(f"{name}/{dataset}: {rw.summarize_reports(WRITTEN[name, dataset])}")

# One report to read (the image inspected in CELL 13 when it has one).
reports = WRITTEN.get((INSPECT_EXPERIMENT, INSPECT_DATASET)) or next(iter(WRITTEN.values()), {})
if reports:
    shown = reports.get(globals().get("image_id")) or reports[next(iter(sorted(reports)))]
    print(f"{shown['image_id']}: verified={shown['verified']} | attempts={len(shown['attempts'])} | "
          f"problems={shown['problems']}")
    display(Markdown(shown["markdown"]))